# Consultas analíticas

**Tech Challenge Fase 3** · Pós-Tech em Data Analytics, FIAP
**Base:** State of Data Brazil, edições 2023-2024, 2024-2025 e 2025-2026
**Etapa do pipeline:** consulta SQL sobre a camada Gold

As consultas abaixo são as mesmas do arquivo `sql/consultas_athena.sql`, executadas aqui em Spark SQL para comprovar que funcionam sobre os dados reais. No AWS Academy Lab elas rodam no Amazon Athena, sobre as tabelas registradas no Glue Data Catalog.

## 1. Configuração e registro das tabelas

In [1]:
import sys, os
os.environ.pop("JAVA_TOOL_OPTIONS", None)
sys.path.insert(0, "../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("tc3")
    .master("local[2]")                      # no AWS Glue esta linha nao existe
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

26/09/05 21:54:23 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/09/05 21:54:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/05 21:54:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3


In [2]:
for tabela in ["gold_distribuicoes", "gold_cruzamentos", "gold_salario", "gold_mencoes"]:
    spark.read.parquet(f"../dados/gold/{tabela}").createOrReplaceTempView(tabela)
    print(f"registrada: {tabela}")

registrada: gold_distribuicoes


registrada: gold_cruzamentos


registrada: gold_salario
registrada: gold_mencoes


## 2. R21. Qual é o índice de adoção de inteligência artificial?

In [3]:
spark.sql("""
    SELECT edicao,
           ROUND(SUM(CASE WHEN categoria LIKE 'Sim,%' THEN participacao_pct ELSE 0 END), 2) AS pct_ia_e_prioridade,
           ROUND(SUM(CASE WHEN categoria LIKE 'Não é uma iniciativa%' THEN participacao_pct ELSE 0 END), 2) AS pct_ia_nao_e_prioridade
    FROM gold_distribuicoes
    WHERE dimensao = 'ia_prioridade'
    GROUP BY edicao
    ORDER BY edicao
""").show()

+---------+-------------------+-----------------------+
|   edicao|pct_ia_e_prioridade|pct_ia_nao_e_prioridade|
+---------+-------------------+-----------------------+
|2023-2024|              36.16|                  28.79|
|2024-2025|              53.59|                  14.83|
|2025-2026|              60.58|                  11.35|
+---------+-------------------+-----------------------+



**Leitura do resultado.** A IA generativa passou de prioridade declarada por 36,2% dos respondentes para 60,6% em duas edições, enquanto a rejeição caiu de 28,8% para 11,4%. É a maior mudança observada em todo o período. A pergunta é respondida apenas por quem trabalha em empresa, com bases de 896, 1.045 e 652 respostas, portanto menores que as da edição. O enunciado pede também o impacto, tratado na consulta seguinte.

### R21, segunda parte. E qual é o impacto declarado?

Prioridade não é resultado. A edição 2025-2026 pergunta pelo estágio dos projetos de IA generativa, com 646 respostas válidas.

In [4]:
spark.sql("""
    SELECT categoria AS estagio_declarado,
           respondentes,
           participacao_pct
    FROM gold_distribuicoes
    WHERE dimensao = 'ia_resultados' AND edicao = '2025-2026'
    ORDER BY participacao_pct DESC
""").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------+------------+----------------+
|estagio_declarado                                                                                                                         |respondentes|participacao_pct|
+------------------------------------------------------------------------------------------------------------------------------------------+------------+----------------+
|Em partes. Temos alguns projetos "piloto", envolvendo AI Generativa e Modelos LLM, rodando mas sem muitos resultados e impacto no negócio.|248         |38.39           |
|Sim. Temos projetos envolvendo AI Generativa e Modelos LLM em produção, gerando resultados e impacto no negócio.                          |171         |26.47           |
|Não. Os projetos envolvendo AI Generativa e Modelos LLM, ainda estão em fase de investigação e planejamento.                              |97   

**Leitura do resultado.** Enquanto 60,6% declaram a IA como prioridade, apenas 26,5% dizem ter projetos em produção gerando resultado no negócio, e 38,4% mantêm pilotos rodando sem impacto declarado. Somados aos 15,0% ainda em planejamento, mais da metade do mercado não saiu da experimentação. A vantagem competitiva deixou de estar em adotar IA e passou a estar em operacionalizá-la.

## 3. R18. Quais perfis são mais valorizados?

In [5]:
spark.sql("""
    SELECT categoria AS nivel,
           MAX(CASE WHEN edicao = '2023-2024' THEN mediana_faixa_reais END) AS mediana_2023_2024,
           MAX(CASE WHEN edicao = '2024-2025' THEN mediana_faixa_reais END) AS mediana_2024_2025,
           MAX(CASE WHEN edicao = '2025-2026' THEN mediana_faixa_reais END) AS mediana_2025_2026
    FROM gold_salario
    WHERE recorte = 'nivel'
    GROUP BY categoria
    ORDER BY mediana_2025_2026
""").show()

+-------------------+-----------------+-----------------+-----------------+
|              nivel|mediana_2023_2024|mediana_2024_2025|mediana_2025_2026|
+-------------------+-----------------+-----------------+-----------------+
|             Júnior|             3500|             3500|             3500|
|              Pleno|             7000|             7000|             7000|
|             Sênior|            10000|            14000|            14000|
|Especialista/Staff+|             NULL|             NULL|            18000|
+-------------------+-----------------+-----------------+-----------------+



**Leitura do resultado.** O nível Especialista/Staff+ aparece apenas em 2025-2026, com faixa mediana de R$ 18.000. É categoria nova do questionário, não comparável com as edições anteriores, e por isso é sempre apresentada em separado.

## 4. R22. Existem diferenças entre modelos de trabalho?

In [6]:
spark.sql("""
    SELECT categoria AS modelo_de_trabalho,
           MAX(CASE WHEN edicao = '2023-2024' THEN participacao_pct END) AS pct_2023_2024,
           MAX(CASE WHEN edicao = '2024-2025' THEN participacao_pct END) AS pct_2024_2025,
           MAX(CASE WHEN edicao = '2025-2026' THEN participacao_pct END) AS pct_2025_2026
    FROM gold_distribuicoes
    WHERE dimensao = 'modelo_trabalho'
    GROUP BY categoria
    ORDER BY pct_2025_2026 DESC
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------+-------------+-------------+-------------+
|modelo_de_trabalho                                                                                            |pct_2023_2024|pct_2024_2025|pct_2025_2026|
+--------------------------------------------------------------------------------------------------------------+-------------+-------------+-------------+
|Modelo 100% remoto                                                                                            |46.31        |45.69        |39.7         |
|Modelo 100% presencial                                                                                        |16.62        |16.29        |20.76        |
|Modelo híbrido com dias fixos de trabalho presencial                                                          |16.62        |17.49        |19.99        |
|Modelo híbrido flexível (o funcionário tem liberdade para escolher qu

**Leitura do resultado.** O trabalho totalmente remoto caiu de 46,3% para 39,7% e o totalmente presencial subiu de 16,6% para 20,8%. O movimento de retorno ao escritório é real e recente, concentrado na última edição.

## 5. R23. Satisfação por modelo de trabalho

In [7]:
spark.sql("""
    SELECT categoria_1 AS modelo_de_trabalho,
           ROUND(MAX(CASE WHEN categoria_2 = 'Sim' THEN participacao_pct END), 2) AS pct_satisfeitos,
           MAX(total_no_grupo) AS respondentes_no_modelo
    FROM gold_cruzamentos
    WHERE dimensao_1 = 'modelo_trabalho' AND dimensao_2 = 'satisfacao' AND edicao = '2025-2026'
    GROUP BY categoria_1
    ORDER BY pct_satisfeitos DESC
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------+---------------+----------------------+
|modelo_de_trabalho                                                                                            |pct_satisfeitos|respondentes_no_modelo|
+--------------------------------------------------------------------------------------------------------------+---------------+----------------------+
|Modelo híbrido flexível (o funcionário tem liberdade para escolher quando estar no escritório presencialmente)|74.96          |631                   |
|Modelo 100% remoto                                                                                            |74.47          |1281                  |
|Modelo híbrido com dias fixos de trabalho presencial                                                          |67.75          |645                   |
|Modelo 100% presencial                                                                 

**Leitura do resultado.** Remoto integral e híbrido flexível empatam no topo, com 74,5% e 75,0% de satisfeitos. A diferença de 0,5 ponto tem z igual a 0,23 e não é significativa a 95% de confiança, portanto afirmar que o remoto lidera seria ler ruído como resultado. O que separa satisfeitos de insatisfeitos é a presença de flexibilidade: são 20,9 pontos entre o híbrido flexível e o presencial integral, que fica em 54,0%. Para uma empresa que precisa reter talento escasso, a política de trabalho deixa de ser assunto administrativo e passa a ser fator de retenção.